In [2]:
import numpy as np
import scipy.stats as stats

np.random.seed(52)

N = 100
beta = 0.95
theta_real = 52

data = stats.uniform(loc=theta_real, scale=theta_real).rvs(size=N)

avg = data.mean()
mx = data.max()

theta_mom = (2 * avg) / 3
theta_mle = (mx * (N + 1)) / (2 * N + 1)

disp = sum((data - avg) ** 2) / N

left = mx / ( ((1 + beta)/2)**(1/N) + 1 )
right = mx / ( ((1 - beta)/2)**(1/N) + 1 )

print("Точный интервал:")
print(f"{left} < θ < {right}")
print("Размер:", right - left)

z1 = stats.norm.ppf((1 - beta)/2)
z2 = stats.norm.ppf((1 + beta)/2)

shift = (2/3) * np.sqrt(disp / N)

a_left = theta_mom - shift * z2
a_right = theta_mom - shift * z1

print("Асимптотический интервал:")
print(f"{a_left} < θ < {a_right}")
print("Размер:", a_right - a_left)

B = 1000

def boot_mom_func(arr):
    res = []
    n_local = len(arr)
    for i in range(B):
        sample = np.random.choice(arr, n_local, True)
        val = (2/3)*sample.mean() - theta_mom
        res.append(val)
    return np.array(sorted(res))

boot_vals = boot_mom_func(data)

l_id = int((1 - beta)/2 * B - 1)
r_id = int((1 + beta)/2 * B - 1)

b_left = theta_mom - boot_vals[r_id]
b_right = theta_mom - boot_vals[l_id]

print("Bootstrap (ОММ):")
print(f"{b_left} < θ < {b_right}")
print("Размер:", b_right - b_left)

def boot_mle_func(arr):
    res = []
    n_local = len(arr)
    for i in range(B):
        sample = np.random.choice(arr, n_local, True)
        val = ((n_local+1)*sample.max()/(2*n_local+1)) - theta_mle
        res.append(val)
    return np.array(sorted(res))

boot_vals = boot_mle_func(data)

b2_left = theta_mle - boot_vals[r_id]
b2_right = theta_mle - boot_vals[l_id]

print("Bootstrap (ОМП):")
print(f"{b2_left} < θ < {b2_right}")
print("Размер:", b2_right - b2_left)

results = [
    ("Точный", right - left),
    ("Асимптотический", a_right - a_left),
    ("Bootstrap MOM", b_right - b_left),
    ("Bootstrap MLE", b2_right - b2_left)
]

results.sort(key=lambda x: x[1])

print("Сравнение интервалов:")
for i, (name, val) in enumerate(results, 1):
    print(f"{i}. {name} -> {round(val, 3)}")

Точный интервал:
51.85955454701832 < θ < 52.80927924436884
Размер: 0.9497246973505185
Асимптотический интервал:
51.7083781503223 < θ < 55.55679059662567
Размер: 3.848412446303371
Bootstrap (ОММ):
51.82156786103661 < θ < 55.64505505260448
Размер: 3.823487191567864
Bootstrap (ОМП):
52.11096560401313 < θ < 52.71621092260204
Размер: 0.6052453185889064
Сравнение интервалов:
1. Bootstrap MLE -> 0.605
2. Точный -> 0.95
3. Bootstrap MOM -> 3.823
4. Асимптотический -> 3.848
